# S5 Causal Masked Reconstruction Colab

This notebook trains a causal `S5` encoder with masked signal reconstruction on Utah-array cache shards stored in Google Drive.

Default design:

- reconstruct normalized raw patch values in `TX + SBP` space
- causal `S5` backbone
- patch-level masking by default with `patch_size=5`, `patch_stride=5`
- contiguous masked spans with a fixed overall mask ratio
- same held-out `Brain2Text25` frozen phoneme probe workflow used in the other `s5` notebooks


In [ ]:
# Mount Drive and resolve cache / output roots.

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
cache_candidates = [
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1",
]

CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
OUTPUT_ROOT = DRIVE_ROOT / "utah_ssl" / "outputs" / "ssl_experiments" / "masked_reconstruction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT :", DRIVE_ROOT)
print("CACHE_ROOT :", CACHE_ROOT, "| exists:", CACHE_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.exists())

if CACHE_ROOT.exists():
    datasets = sorted(p.name for p in CACHE_ROOT.iterdir() if p.is_dir())
    print("datasets:", datasets)
else:
    print("cache candidates checked:")
    for path in cache_candidates:
        print(" -", path)


In [ ]:
# Clone the public repo and import the reusable masked SSL helpers.

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ethan-read/utah-ssl.git"
REPO_DIR = Path("/content/utah-ssl")
EXPERIMENTS_DIR = REPO_DIR / "analysis" / "active" / "ssl_experiments"
MASKED_SSL_DIR = EXPERIMENTS_DIR / "masked_ssl"
SSL_DIR = REPO_DIR / "analysis" / "active" / "transfer_benchmark" / "ssl_autoresearch"

os.chdir("/content")

if REPO_DIR.exists():
    print("Using existing repo:", REPO_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

for candidate in (REPO_DIR, EXPERIMENTS_DIR, SSL_DIR):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

if not MASKED_SSL_DIR.exists():
    raise FileNotFoundError(
        "The cloned repo does not contain analysis/active/ssl_experiments/masked_ssl. "
        "Make sure REPO_DIR points at a checkout that includes the masked reconstruction package."
    )

from masked_ssl import (
    CacheAccessConfig,
    DownstreamProbeConfig,
    SSLTrainingConfig,
    build_random_init_probe_state,
    build_segment_sampler,
    list_ssl_checkpoints,
    load_precomputed_session_feature_stats_into_cache_context,
    plot_ssl_training_history,
    prepare_cache_context,
    recover_downstream_probe_state,
    recover_ssl_run_state_from_checkpoint,
    resolve_ssl_checkpoint_path,
    run_downstream_probe,
    run_probe_head_sweep,
    run_ssl_training,
)
from masked_ssl.probe import build_downstream_probe_problem

print("cwd:", Path.cwd())
print("repo dir exists:", REPO_DIR.exists(), REPO_DIR)
print("experiments dir exists:", EXPERIMENTS_DIR.exists(), EXPERIMENTS_DIR)
print("masked_ssl dir exists:", MASKED_SSL_DIR.exists(), MASKED_SSL_DIR)
print("ssl dir exists:", SSL_DIR.exists(), SSL_DIR)
print("SSL_AUTORESEARCH_CACHE_ROOT:", os.environ["SSL_AUTORESEARCH_CACHE_ROOT"])
print("SSL_AUTORESEARCH_OUTPUT_ROOT:", os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"])


In [ ]:
# Experiment config.

SEED = 7

SSL_STATE_MODE = "train"  # Set to "recover" to load a previous checkpoint.
STABLE_SSL_SESSION_STATS_PATH = (
    DRIVE_ROOT
    / "utah_ssl"
    / "outputs"
    / "ssl_experiments"
    / "contrastive"
    / "precomputed_ssl_session_stats"
    / "session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt"
)
SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH = None
SSL_RECOVERY_RUN_DIR = None
ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH = None

SEGMENT_BINS = 80
PATCH_SIZE = 5
PATCH_STRIDE = 5
HIDDEN_SIZE = 256
S5_STATE_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 32
NUM_STEPS = 600
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
VAL_EVERY = 50
VAL_BATCHES = 10
CHECKPOINT_EVERY_STEPS = 100
DATASET_WEIGHT_ALPHA = 0.25
EXAMPLES_PER_SHARD = 8
POST_PROJ_NORM = "rms"

MASK_UNIT = "patch"  # Set to "bin" for raw-bin contiguous masking.
MASK_TOKEN_PLACEMENT = "before_projection"
MASK_RATIO = 0.40
PATCH_SPAN_LENGTH_MIN = 2
PATCH_SPAN_LENGTH_MAX = 8
BIN_SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN * PATCH_SIZE
BIN_SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX * PATCH_SIZE
SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MIN
SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MAX
NUM_SPANS_MODE = "one"
ALLOW_BIN_FRACTIONAL_OVERLAP = True
RECONSTRUCT_TARGET = "raw_patch"
LOSS_MODE = "masked_only"

CACHE_ACCESS_MODE = "drive_direct"  # or "copy_to_local"
LOCAL_CACHE_BASE = "/content/utah_ssl_cache"
FORCE_RECOPY_LOCAL_CACHE = False
EXCLUDED_DATASETS = {"brain2text25"}
LOG_EVERY = 10

NORMALIZE_IMPL_VERSION = "session_featurewise_v1"
CACHE_PREPARE_NORMALIZE_IMPL_VERSION = "segment_prefix_v1"
NORMALIZE_CONTEXT_BINS = min(16, SEGMENT_BINS)

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode=CACHE_ACCESS_MODE,
    local_cache_base=LOCAL_CACHE_BASE,
    force_recopy_local_cache=FORCE_RECOPY_LOCAL_CACHE,
    excluded_datasets=tuple(sorted(EXCLUDED_DATASETS)),
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    normalize_context_bins=NORMALIZE_CONTEXT_BINS,
    normalize_impl_version=CACHE_PREPARE_NORMALIZE_IMPL_VERSION,
    examples_per_shard=EXAMPLES_PER_SHARD,
)

SSL_TRAINING_CONFIG = SSLTrainingConfig(
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
    hidden_size=HIDDEN_SIZE,
    s5_state_size=S5_STATE_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_every=VAL_EVERY,
    val_batches=VAL_BATCHES,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
    log_every=LOG_EVERY,
    post_proj_norm=POST_PROJ_NORM,
    mask_unit=MASK_UNIT,
    mask_token_placement=MASK_TOKEN_PLACEMENT,
    mask_ratio=MASK_RATIO,
    span_length_min=SPAN_LENGTH_MIN,
    span_length_max=SPAN_LENGTH_MAX,
    num_spans_mode=NUM_SPANS_MODE,
    reconstruct_target=RECONSTRUCT_TARGET,
    loss_mode=LOSS_MODE,
    allow_bin_fractional_overlap=ALLOW_BIN_FRACTIONAL_OVERLAP,
)

print("Notebook workflow switches:", {
    "SSL_STATE_MODE": SSL_STATE_MODE,
    "STABLE_SSL_SESSION_STATS_PATH": STABLE_SSL_SESSION_STATS_PATH,
    "SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH": SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    "SSL_RECOVERY_RUN_DIR": SSL_RECOVERY_RUN_DIR,
    "ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH": ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH,
})
print("Masking config:", {
    "mask_unit": MASK_UNIT,
    "mask_token_placement": MASK_TOKEN_PLACEMENT,
    "mask_ratio": MASK_RATIO,
    "span_length_min": SPAN_LENGTH_MIN,
    "span_length_max": SPAN_LENGTH_MAX,
    "num_spans_mode": NUM_SPANS_MODE,
})
print("Patch config:", {
    "patch_size": PATCH_SIZE,
    "patch_stride": PATCH_STRIDE,
    "segment_bins": SEGMENT_BINS,
})
print("CACHE_ACCESS_CONFIG:", CACHE_ACCESS_CONFIG)
print("SSL_TRAINING_CONFIG:", SSL_TRAINING_CONFIG)


In [ ]:
# Resolve cache access mode, summarize datasets, and build the reusable cache context.

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CACHE_CONTEXT = prepare_cache_context(
    cache_candidates=cache_candidates,
    config=CACHE_ACCESS_CONFIG,
)
FULL_DIM = CACHE_CONTEXT.full_dim

print("DEVICE:", DEVICE)
print("CACHE_CONTEXT cache_root:", CACHE_CONTEXT.cache_root)
print("FULL_DIM:", FULL_DIM)
print("session split summary:")
for dataset, summary in CACHE_CONTEXT.session_split_summary.items():
    print(
        f" - {dataset}: sessions={summary['total_sessions']} train_sessions={summary['train_sessions']} "
        f"val_sessions={summary['val_sessions']} train_examples={summary['train_examples']} "
        f"val_examples={summary['val_examples']}"
    )


In [ ]:
# Load precomputed SSL session-level featurewise z-scoring stats from the stable Drive path.

SSL_SESSION_STATS_STATE = load_precomputed_session_feature_stats_into_cache_context(
    cache_context=CACHE_CONTEXT,
    stats_path=STABLE_SSL_SESSION_STATS_PATH,
    normalize_impl_version=NORMALIZE_IMPL_VERSION,
)

print("Loaded cached SSL session stats from stable path.")
print("session_count:", SSL_SESSION_STATS_STATE["session_count"])
print("normalize_impl_version:", SSL_SESSION_STATS_STATE["normalize_impl_version"])
print("metadata:", SSL_SESSION_STATS_STATE["metadata"])


In [ ]:
# Quick batch inspection.

INSPECT_SAMPLER = build_segment_sampler(
    CACHE_CONTEXT,
    "train",
    batch_size=4,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
)
inspect_batch = INSPECT_SAMPLER.sample_batch()
print("inspect_batch[x].shape:", tuple(inspect_batch["x"].shape))
print("inspect_batch[lengths]:", inspect_batch["lengths"].tolist())
print("inspect_batch[datasets]:", inspect_batch["datasets"])
print("inspect_batch[sessions]:", inspect_batch["sessions"])


In [ ]:
# Acquire SSL state: either train now or recover from a saved checkpoint.

if NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and not CACHE_CONTEXT.session_feature_stats:
    raise RuntimeError("Run the stable session-stats cell first so session-level z-scoring is loaded into CACHE_CONTEXT.")

if SSL_STATE_MODE == "train":
    SSL_RUN_STATE = run_ssl_training(
        cache_context=CACHE_CONTEXT,
        config=SSL_TRAINING_CONFIG,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
elif SSL_STATE_MODE == "recover":
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
else:
    raise ValueError(f"Unsupported SSL_STATE_MODE: {SSL_STATE_MODE}")

print("run_name:", SSL_RUN_STATE["run_name"])
print("run_dir:", SSL_RUN_STATE["run_dir"])
print("checkpoint_path:", SSL_RUN_STATE["checkpoint_path"])
print("best_score:", SSL_RUN_STATE["best_score"])
print("best_step:", SSL_RUN_STATE["best_step"])


In [ ]:
# Resolve the active checkpoint path used by downstream probe recovery.

if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(SSL_RUN_STATE["checkpoint_path"])
else:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
    if not ACTIVE_SSL_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {ACTIVE_SSL_CHECKPOINT_PATH}"
        )

ACTIVE_SSL_CHECKPOINT_RUN_DIR = (
    ACTIVE_SSL_CHECKPOINT_PATH.parent.parent
    if ACTIVE_SSL_CHECKPOINT_PATH.parent.name == "checkpoints"
    else ACTIVE_SSL_CHECKPOINT_PATH.parent
)

print("ACTIVE_SSL_CHECKPOINT_PATH:", ACTIVE_SSL_CHECKPOINT_PATH)
print("ACTIVE_SSL_CHECKPOINT_RUN_DIR:", ACTIVE_SSL_CHECKPOINT_RUN_DIR)


In [ ]:
# Plot train / val curves for the active run.

SSL_PLOT_STATE = plot_ssl_training_history(SSL_RUN_STATE)
print("latest train record:", SSL_RUN_STATE["train_history"][-1] if SSL_RUN_STATE["train_history"] else None)
print("latest val record:", SSL_RUN_STATE["val_history"][-1] if SSL_RUN_STATE["val_history"] else None)


In [ ]:
# Optional: list saved SSL checkpoints for the active run.

import pandas as pd

CHECKPOINT_LIST_RUN_DIR = Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR)
AVAILABLE_SSL_CHECKPOINTS = list_ssl_checkpoints(CHECKPOINT_LIST_RUN_DIR)
display(pd.DataFrame(AVAILABLE_SSL_CHECKPOINTS))


## Downstream Probe Workflow

These cells keep the same held-out `Brain2Text25` workflow as the other `s5` notebooks.
The main apples-to-apples frozen comparison is:

- pretrained masked-reconstruction encoder, frozen, linear probe
- random-init encoder, frozen, same linear probe


In [ ]:
# Configure and preview the held-out Brain2Text25 probe split.

PROBE_SESSION_LIMIT = 8
PROBE_TARGET_SESSION_COUNT = 4
PROBE_BATCH_SIZE = 8
PROBE_BUDGET_SECONDS = 240
PROBE_MAX_STEPS = 400
PROBE_HEAD_LEARNING_RATE = 1e-3
ENCODER_LEARNING_RATE = 3e-4
PROBE_WEIGHT_DECAY = 1e-2

DOWNSTREAM_PROBE_CONFIG = DownstreamProbeConfig(
    enabled=True,
    seed=SEED,
    session_limit=PROBE_SESSION_LIMIT,
    target_session_count=PROBE_TARGET_SESSION_COUNT,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_budget_seconds=PROBE_BUDGET_SECONDS,
    max_probe_steps=PROBE_MAX_STEPS,
    probe_head_learning_rate=PROBE_HEAD_LEARNING_RATE,
    encoder_learning_rate=None,
    weight_decay=PROBE_WEIGHT_DECAY,
    probe_head_type="linear",
)

downstream_problem_preview = build_downstream_probe_problem(
    cache_root=Path(CACHE_CONTEXT.cache_root),
    probe_config=DOWNSTREAM_PROBE_CONFIG,
)
print("eligible_sessions:", len(downstream_problem_preview["eligible_entries"]))
print("preview_source_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].train])
print("preview_target_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].val])


In [ ]:
# Shared downstream experiment helpers.

import pandas as pd

FROZEN_LINEAR_PROBE_OVERRIDES = {"probe_head_type": "linear"}
QUICK_CONV1D_PROBE_OVERRIDES = {
    "probe_head_type": "conv1d",
    "probe_conv_hidden_size": HIDDEN_SIZE,
    "probe_conv_kernel_size": 3,
    "probe_budget_seconds": 120,
    "max_probe_steps": 50,
}
B2_LINEAR_PROBE_OVERRIDES = {
    "probe_head_type": "linear",
    "probe_head_learning_rate": PROBE_HEAD_LEARNING_RATE,
    "encoder_learning_rate": ENCODER_LEARNING_RATE,
    "weight_decay": PROBE_WEIGHT_DECAY,
    "probe_budget_seconds": 600,
    "max_probe_steps": 800,
}

def build_notebook_random_probe_state(*, seed_offset: int = 0):
    return build_random_init_probe_state(
        reference_config=DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_config"],
        input_dim=FULL_DIM,
        seed=SEED + int(seed_offset),
        base_run_dir=DOWNSTREAM_PROBE_BASE_RUN_DIR,
    )

def display_probe_summaries(*summary_dicts):
    rows = [summary for summary in summary_dicts if summary is not None]
    if not rows:
        print("No summaries to display.")
    else:
        display(pd.DataFrame(rows))


In [ ]:
# Recover the default SSL encoder and prepare reusable downstream probe state.

DOWNSTREAM_PROBE_DEFAULT_STATE = recover_downstream_probe_state(
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    output_root=OUTPUT_ROOT,
    input_dim=FULL_DIM,
    default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
    in_memory_model=SSL_RUN_STATE["model"] if "SSL_RUN_STATE" in globals() else None,
    current_checkpoint_path=Path(ACTIVE_SSL_CHECKPOINT_PATH),
    current_run_dir=Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR),
)

DOWNSTREAM_PROBE_BASE_RUN_DIR = Path(DOWNSTREAM_PROBE_DEFAULT_STATE["base_run_dir"])
print("DOWNSTREAM_PROBE_ENCODER_SOURCE:", DOWNSTREAM_PROBE_DEFAULT_STATE["source"])
print("DOWNSTREAM_PROBE_BASE_RUN_DIR:", DOWNSTREAM_PROBE_BASE_RUN_DIR)
print("DOWNSTREAM_PROBE_CHECKPOINT_PATH:", DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_path"])


In [ ]:
# Frozen SSL checkpoint linear probe.

SSL_CHECKPOINT_LINEAR_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    probe_overrides=FROZEN_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_LINEAR_SUMMARY)


In [ ]:
# Random-init frozen linear probe baseline.

RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=1000)
RANDOM_INIT_LINEAR_SUMMARY = run_downstream_probe(
    probe_state=RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init",
    artifact_prefix="random_init",
    train_encoder=False,
    probe_overrides=FROZEN_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_LINEAR_SUMMARY, RANDOM_INIT_LINEAR_SUMMARY)


In [ ]:
# Optional stronger frozen-head diagnostic.

SSL_CHECKPOINT_CONV1D_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    probe_overrides=QUICK_CONV1D_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_CONV1D_SUMMARY)


In [ ]:
# Full fine-tuning from the masked SSL checkpoint.

SSL_CHECKPOINT_B2_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint_b2",
    artifact_prefix="ssl_checkpoint_b2",
    train_encoder=True,
    probe_overrides=B2_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_B2_SUMMARY)


In [ ]:
# Full fine-tuning from random initialization.

RANDOM_INIT_B2_STATE = build_notebook_random_probe_state(seed_offset=2000)
RANDOM_INIT_B2_SUMMARY = run_downstream_probe(
    probe_state=RANDOM_INIT_B2_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init_b2",
    artifact_prefix="random_init_b2",
    train_encoder=True,
    probe_overrides=B2_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_B2_SUMMARY, RANDOM_INIT_B2_SUMMARY)


In [ ]:
# Compact comparison table for the main summaries collected above.

MAIN_SUMMARIES = [
    globals().get("SSL_CHECKPOINT_LINEAR_SUMMARY"),
    globals().get("RANDOM_INIT_LINEAR_SUMMARY"),
    globals().get("SSL_CHECKPOINT_B2_SUMMARY"),
    globals().get("RANDOM_INIT_B2_SUMMARY"),
]
display_probe_summaries(*MAIN_SUMMARIES)


## Notes

This notebook intentionally omits the contrastive retrieval diagnostics from the older `s5` notebooks.
For this experiment, the primary scorecard is held-out frozen phoneme transfer, with masked-reconstruction loss used as a training diagnostic.
